# Stage 4.5 v4 — TrackNet ball detector (self-contained)

**One file. Upload this notebook, Run All.** Mixed precision (AMP) is on and
`BATCH` is preset to fit the A100 — nothing to edit.

Needs only `MyDrive/pb_v4_upload.zip` (already uploaded). Saves
`ball_model_v4.pt` + `validation_report.json` to Drive.

Split: train pb_3/4/5min (same court) · val pb_2min · **test pb_3min_court2
(different court = generalization)**.

To change batch size, edit `BATCH` in the KNOBS cell (lower to 2 if OOM).
**Runtime -> Change runtime type -> GPU.**

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # before torch/CUDA

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import zipfile, sys
from pathlib import Path
LOCAL = Path('/content/pb_v4')
if not LOCAL.exists():
    print('unzipping bundle to local disk...')
    with zipfile.ZipFile('/content/drive/MyDrive/pb_v4_upload.zip') as z:
        z.extractall(LOCAL)
REPO = LOCAL/'repo'; DATA = LOCAL/'data'
sys.path.insert(0, str(REPO))
print('clips:', sorted(p.name for p in DATA.iterdir()))

In [ ]:
try:
    import cv2
except Exception:
    !pip -q install opencv-python-headless
    import cv2

In [ ]:
# ===== KNOBS (edit if needed) =====
import torch as _t
if _t.cuda.is_available():
    _gb = _t.cuda.get_device_properties(0).total_memory / 1e9
    BATCH = 2 if _gb < 18 else (4 if _gb < 34 else (8 if _gb < 50 else 12))  # T4->2 L4/A10->4 A100-40->8 A100-80->12
    print(f'GPU {_gb:.0f} GB -> auto BATCH {BATCH}')
else:
    BATCH = 2
EPOCHS = 15
LR     = 5e-4
ACCUM  = 4        # grad accumulation -> effective batch = BATCH*ACCUM (stability, no extra mem)
PROC_H, PROC_W = 720, 1280
TOL, CONF, SIGMA = 6, 0.30, 3.0

import json, numpy as np, torch, random
torch.manual_seed(0); np.random.seed(0); random.seed(0)  # reproducible
from torch.utils.data import Dataset, DataLoader
from stages.track_ball._tracknet_model import TrackNet   # the only bundle import (stable)

def make_heatmap(x, y):
    hm = np.zeros((PROC_H, PROC_W), np.float32)
    if x is None or y is None: return hm
    ix, iy = int(round(x)), int(round(y))
    if not (0 <= ix < PROC_W and 0 <= iy < PROC_H): return hm
    r = int(3*SIGMA)
    x0, x1 = max(0, ix-r), min(PROC_W, ix+r+1)
    y0, y1 = max(0, iy-r), min(PROC_H, iy+r+1)
    xs = np.arange(x0, x1)[None, :]; ys = np.arange(y0, y1)[:, None]
    hm[y0:y1, x0:x1] = np.exp(-((xs-ix)**2 + (ys-iy)**2)/(2*SIGMA**2)).astype(np.float32)
    hm[iy, ix] = 1.0
    return hm

def focal(pred, gt, a=2.0, b=4.0, eps=1e-6):
    pred = pred.clamp(eps, 1-eps)
    pos = gt.eq(1.0).float(); neg = 1 - pos
    pl = ((1-pred)**a) * torch.log(pred) * pos
    nl = ((1-gt)**b) * (pred**a) * torch.log(1-pred) * neg
    return -(pl.sum() + nl.sum()) / pos.sum().clamp(min=1.0)

class DS(Dataset):
    def __init__(self, folders, aug=False):
        self.items = []; self.aug = aug
        for f in folders:
            m = json.load(open(f/'v4_manifest.json'))
            fd = f/m.get('frames_dir', 'frames_720')
            for s in m['samples']: self.items.append((s, fd))
    def __len__(self): return len(self.items)
    def rd(self, fd, i):
        im = cv2.imread(str(fd/f'{i}.jpg'))
        if im is None: return None
        if im.shape[:2] != (PROC_H, PROC_W): im = cv2.resize(im, (PROC_W, PROC_H))
        return cv2.cvtColor(im, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
    def __getitem__(self, k):
        s, fd = self.items[k]
        frs = [self.rd(fd, i) for i in s['frames']]
        if any(f is None for f in frs):
            return (torch.zeros(9, PROC_H, PROC_W), torch.zeros(1, PROC_H, PROC_W), -1.0, -1.0)
        st = np.concatenate([f.transpose(2, 0, 1) for f in frs], 0).astype(np.float32)
        if s['visible']:
            tg = make_heatmap(s['x_proc'], s['y_proc'])[None]
            px, py = float(s['x_proc']), float(s['y_proc'])
        else:
            tg = np.zeros((1, PROC_H, PROC_W), np.float32); px = py = -1.0
        if self.aug:
            if np.random.rand() < 0.5:
                st = st[:, :, ::-1].copy(); tg = tg[:, :, ::-1].copy()
                if px >= 0: px = PROC_W - 1 - px
            if np.random.rand() < 0.7:
                g = 1 + (np.random.rand()-0.5)*0.4; bs = (np.random.rand()-0.5)*0.1
                st = np.clip(st*g + bs, 0, 1).astype(np.float32)
        return torch.from_numpy(st), torch.from_numpy(tg.astype(np.float32)), px, py

def peak(hm):
    iy, ix = np.unravel_index(int(hm.argmax()), hm.shape)
    return ix, iy, float(hm[iy, ix])

@torch.no_grad()
def evaluate(model, ds, dev):
    model.eval(); dl = DataLoader(ds, batch_size=BATCH, num_workers=2)
    tp = fn = fp = tn = 0
    for st, tg, px, py in dl:
        with torch.cuda.amp.autocast(enabled=dev == 'cuda'):
            out = model(st.to(dev))
        pr = out.float().cpu().numpy()
        for j in range(pr.shape[0]):
            x, y, v = peak(pr[j, 0]); det = v >= CONF
            gx, gy = float(px[j]), float(py[j])
            if gx >= 0:
                ok = det and (np.hypot(x-gx, y-gy) <= TOL)
                tp += int(ok); fn += int(not ok)
            else:
                fp += int(det); tn += int(not det)
    return tp/max(tp+fn, 1), fp/max(fp+tn, 1)

print('training utils defined; BATCH =', BATCH)

In [ ]:
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
train = DS([DATA/'pb_3min', DATA/'pb_4min', DATA/'pb_5min'], aug=True)
val   = DS([DATA/'pb_2min'])
test  = DS([DATA/'pb_3min_court2'])
print('train', len(train), '| val', len(val), '| test', len(test))

model = TrackNet(in_channels=9, out_channels=1, input_shape=(PROC_H, PROC_W)).to(dev)
opt   = torch.optim.Adam(model.parameters(), lr=LR)
scaler = torch.cuda.amp.GradScaler(enabled=dev == 'cuda')

def _fits(bs):
    try:
        torch.cuda.empty_cache()
        st = torch.zeros(bs, 9, PROC_H, PROC_W, device=dev)
        tg = torch.zeros(bs, 1, PROC_H, PROC_W, device=dev)
        with torch.cuda.amp.autocast(enabled=dev == 'cuda'):
            out = model(st)
        focal(out.float(), tg).backward()
        model.zero_grad(set_to_none=True)
        del st, tg, out
        torch.cuda.empty_cache()
        return True
    except RuntimeError as e:
        model.zero_grad(set_to_none=True)
        torch.cuda.empty_cache()
        if 'out of memory' in str(e).lower():
            return False
        raise
if dev == 'cuda':
    while BATCH > 1 and not _fits(BATCH):
        print(f'  BATCH {BATCH} OOM -> trying {BATCH // 2}')
        BATCH = BATCH // 2
print(f'using BATCH {BATCH} x ACCUM {ACCUM} = effective {BATCH*ACCUM}')

sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
dl = DataLoader(train, batch_size=BATCH, shuffle=True, num_workers=2)

best = -1.0
bvr = bvfp = btr = btfp = 0.0
for ep in range(EPOCHS):
    model.train(); tot = 0.0; opt.zero_grad()
    for i, (st, tg, _, _) in enumerate(dl):
        with torch.cuda.amp.autocast(enabled=dev == 'cuda'):
            pred = model(st.to(dev))
        loss = focal(pred.float(), tg.to(dev).float()) / ACCUM
        scaler.scale(loss).backward()
        if (i + 1) % ACCUM == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt); scaler.update(); opt.zero_grad()
        tot += loss.item() * ACCUM
    sched.step()
    vr, vfp = evaluate(model, val, dev)
    score = vr - vfp
    msg = f'epoch {ep}: loss {tot/len(dl):.3f}  val_recall {vr:.3f} fp {vfp:.3f}  score {score:.3f}'
    if score > best:
        best = score; bvr, bvfp = vr, vfp
        tr, tfp = evaluate(model, test, dev)
        btr, btfp = tr, tfp
        torch.save({'state_dict': model.state_dict(), 'input_shape': (PROC_H, PROC_W),
                    'in_channels': 9, 'out_channels': 1,
                    'val_recall': vr, 'val_fp': vfp, 'test_recall': tr, 'test_fp': tfp,
                    'val_score': score},
                   '/content/drive/MyDrive/ball_model_v4.pt')
        json.dump({'best_val_score': best, 'val_recall': vr, 'val_fp': vfp,
                   'test_recall': tr, 'test_fp': tfp,
                   'val_clip': 'pb_2min', 'test_clip': 'pb_3min_court2'},
                  open('/content/drive/MyDrive/validation_report.json', 'w'), indent=1)
        msg += f'  |  TEST_recall {tr:.3f} fp {tfp:.3f}  <- saved'
    print(msg)
print(f'DONE. best val_recall {bvr:.3f} (fp {bvfp:.3f}) | test_recall {btr:.3f} (fp {btfp:.3f})')

## Done
- **val recall** (same court, new players) — easy case, should be high.
- **test recall** (different court) — the number that matters; **bar >= 0.80**.
  - both high -> generalizes. val high / test low -> overfit to court (add
    court2 to training). both low -> bump to 1080p.
- Download `MyDrive/ball_model_v4.pt` -> `data/models/` and report both.